In [7]:
import numpy as np
import matplotlib.pyplot as plt

In [8]:
"""
Load SIS simulation event times output by the Java App into a NumPy array.

- Input directory: `output/sis`
  - `metadata.csv` for parameters (itr, batch_num, c_list, lambda range)
  - `times_XX.txt` where each line is a comma-separated list of event times

- Output array `time`: shape = (len(cList), len(lambdaList), itr), dtype=object
  Each element is a 1D NumPy array of float64 event times for a single run.

Usage:
    from scripts.load_sis_time import load_time_array
    time, axes = load_time_array(output_dir="output/sis", batch_idx=0)
    # time.shape -> (C, L, itr)
    # axes["c_list"], axes["lambda_list"], axes["itr"]
"""

from __future__ import annotations

import os
import sys
import numpy as np
from typing import Dict, Tuple

output_path = os.path.abspath(os.path.join('..', 'output/sis'))
# output_path = os.path.abspath(os.path.join('..', 'java-project', 'output'))
if output_path not in sys.path:
    sys.path.append(output_path)


def _read_metadata(meta_path: str) -> Dict[str, str]:
    """Read key,value style metadata where values may contain commas.

    This parser splits each line at the first comma only.
    Returns a dict of raw string values.
    """
    meta: Dict[str, str] = {}
    with open(meta_path, "r", encoding="utf-8") as f:
        # skip header if present
        header = f.readline()
        if "," not in header:
            # Unexpected; treat as data line
            key, val = header.strip().split(",", 1)
            meta[key.strip()] = val.strip()
        for line in f:
            line = line.rstrip("\n")
            if not line:
                continue
            if "," not in line:
                continue
            key, val = line.split(",", 1)
            meta[key.strip()] = val.strip()
    return meta


def _build_lambda_list(meta: Dict[str, str]) -> np.ndarray:
    lam_min = float(meta["lambda_min"]) if "lambda_min" in meta else float(meta["lambdaMin"])
    lam_max = float(meta["lambda_max"]) if "lambda_max" in meta else float(meta["lambdaMax"])
    if "lambda_count" in meta:
        count = int(meta["lambda_count"])
        # Inclusive endpoints
        lam = np.linspace(lam_min, lam_max, count)
    else:
        step = float(meta["dlambda"]) if "dlambda" in meta else float(meta["dLambda"])  # fallback key
        # Replicate Java's inclusive arange: ((end-start)/step) + 1
        count = int(round((lam_max - lam_min) / step)) + 1
        lam = lam_min + step * np.arange(count)
    # Reduce tiny FP noise
    return np.round(lam, 10)


def _parse_c_list(meta: Dict[str, str]) -> np.ndarray:
    # In metadata.csv: key is "c_list"; value like "0.0,0.4,1.2"
    # Fall back to params.csv-style key if needed.
    raw = meta.get("c_list") or meta.get("cList")
    if raw is None:
        raise ValueError("c_list not found in metadata")
    return np.array([float(x) for x in raw.split(",") if x.strip() != ""], dtype=float)


def load_time_array(output_dir: str = output_path, batch_idx: int = 0) -> Tuple[np.ndarray, Dict[str, object]]:
    """Load times for one batch into a 3D object array.

    Parameters
    - output_dir: directory containing `metadata.csv` and `times_XX.txt` files
    - batch_idx: which `times_XX.txt` to load (0-based)

    Returns
    - time: np.ndarray with shape (C, L, itr), dtype=object
    - axes: dict with keys: "c_list" (np.ndarray), "lambda_list" (np.ndarray), "itr" (int), "batch_num" (int)
    """

    print(f"Loading times from {output_dir} batch {batch_idx}")

    meta_path = os.path.join(output_dir, "metadata.csv")
    if not os.path.exists(meta_path):
        raise FileNotFoundError(f"metadata.csv not found in {output_dir}")

    meta = _read_metadata(meta_path)
    c_list = _parse_c_list(meta)
    lambda_list = _build_lambda_list(meta)
    itr = int(meta.get("itr") or meta.get("Itr") or 0)
    batch_num = int(meta.get("batch_num") or meta.get("batchNum") or 1)

    times_file = os.path.join(output_dir, f"times_{batch_idx:02d}.txt")
    if not os.path.exists(times_file):
        raise FileNotFoundError(f"Times file not found: {times_file}")

    C = len(c_list)
    L = len(lambda_list)
    expected_lines = C * L * itr

    time = np.empty((C, L, itr), dtype=object)

    # Read line-by-line to avoid excessive memory usage.
    line_count = 0
    with open(times_file, "r", encoding="utf-8") as f:
        for line_idx, line in enumerate(f):
            s = line.strip()
            if not s:
                arr = np.array([], dtype=float)
            else:
                # Fast numeric parsing
                arr = np.fromstring(s, sep=",", dtype=float)

            if line_idx >= expected_lines:
                # Extra lines present; stop once we've filled the array
                break

            # Map flat index -> (cIdx, lIdx, itIdx)
            cIdx = line_idx // (L * itr)
            rem = line_idx % (L * itr)
            lIdx = rem // itr
            itIdx = rem % itr
            time[cIdx, lIdx, itIdx] = arr
            line_count += 1

    if line_count != expected_lines:
        raise ValueError(
            f"Unexpected number of lines in {os.path.basename(times_file)}: "
            f"got {line_count}, expected {expected_lines} (C={C}, L={L}, itr={itr})"
        )

    axes = {
        "c_list": c_list,
        "lambda_list": lambda_list,
        "itr": itr,
        "batch_num": batch_num,
    }
    return time, axes

def load_infected_array(output_dir: str = output_path, batch_idx: int = 0) -> Tuple[np.ndarray, Dict[str, object]]:
    """Load times for one batch into a 3D object array.

    Parameters
    - output_dir: directory containing `metadata.csv` and `times_XX.txt` files
    - batch_idx: which `infected_num_XX.txt` to load (0-based)

    Returns
    - time: np.ndarray with shape (C, L, itr), dtype=int
    - axes: dict with keys: "c_list" (np.ndarray), "lambda_list" (np.ndarray), "itr" (int), "batch_num" (int)
    """

    print(f"Loading infected_num from {output_dir} batch {batch_idx}")

    meta_path = os.path.join(output_dir, "metadata.csv")
    if not os.path.exists(meta_path):
        raise FileNotFoundError(f"metadata.csv not found in {output_dir}")

    meta = _read_metadata(meta_path)
    c_list = _parse_c_list(meta)
    lambda_list = _build_lambda_list(meta)
    itr = int(meta.get("itr") or meta.get("Itr") or 0)
    batch_num = int(meta.get("batch_num") or meta.get("batchNum") or 1)

    infected_num_file = os.path.join(output_dir, f"infected_num_{batch_idx:02d}.txt")
    if not os.path.exists(infected_num_file):
        raise FileNotFoundError(f"infected_num file not found: {infected_num_file}")

    C = len(c_list)
    L = len(lambda_list)
    expected_lines = C * L * itr

    infected_num = np.empty((C, L, itr), dtype=int)

    # Read line-by-line to avoid excessive memory usage.
    line_count = 0
    with open(infected_num_file, "r", encoding="utf-8") as f:
        for line_idx, line in enumerate(f):
            s = line.strip()
            if not s:
                arr = np.array([], dtype=float)
            else:
                # Fast numeric parsing
                arr = np.fromstring(s, sep=",", dtype=float)

            if line_idx >= expected_lines:
                # Extra lines present; stop once we've filled the array
                break

            # Map flat index -> (cIdx, lIdx, itIdx)
            cIdx = line_idx // (L * itr)
            rem = line_idx % (L * itr)
            lIdx = rem // itr
            itIdx = rem % itr
            infected_num[cIdx, lIdx, itIdx] = arr
            line_count += 1

    if line_count != expected_lines:
        raise ValueError(
            f"Unexpected number of lines in {os.path.basename(infected_num_file)}: "
            f"got {line_count}, expected {expected_lines} (C={C}, L={L}, itr={itr})"
        )

    axes = {
        "c_list": c_list,
        "lambda_list": lambda_list,
        "itr": itr,
        "batch_num": batch_num,
    }
    return infected_num, axes

In [9]:
# Quick sanity check: load the first batch and print a brief summary.
t, axes = load_time_array()
C, L, itr = t.shape
print(f"Loaded times: shape={t.shape}, dtype={t.dtype}")
print(f"c_list ({C}): {axes['c_list']}")
print(f"lambda_list ({L}): {axes['lambda_list'][0]} .. {axes['lambda_list'][-1]} (len={L})")

time_all = np.concatenate([load_time_array(batch_idx=b)[0] for b in range(axes['batch_num'])], axis=2)
infected_num_all = np.concatenate([load_time_array(batch_idx=b)[0] for b in range(axes['batch_num'])], axis=2)

print(time_all.shape)
print(infected_num_all.shape)

Loading times from /Users/penguin/trend-sar/output/sis batch 0
Loaded times: shape=(3, 21, 10), dtype=object
c_list (3): [0.  0.4 1.2]
lambda_list (21): 0.0 .. 0.2 (len=21)
Loading times from /Users/penguin/trend-sar/output/sis batch 0
Loading times from /Users/penguin/trend-sar/output/sis batch 1
Loading times from /Users/penguin/trend-sar/output/sis batch 2
Loading times from /Users/penguin/trend-sar/output/sis batch 3
Loading times from /Users/penguin/trend-sar/output/sis batch 4
Loading times from /Users/penguin/trend-sar/output/sis batch 5
Loading times from /Users/penguin/trend-sar/output/sis batch 6
Loading times from /Users/penguin/trend-sar/output/sis batch 0
Loading times from /Users/penguin/trend-sar/output/sis batch 1
Loading times from /Users/penguin/trend-sar/output/sis batch 2
Loading times from /Users/penguin/trend-sar/output/sis batch 3
Loading times from /Users/penguin/trend-sar/output/sis batch 4
Loading times from /Users/penguin/trend-sar/output/sis batch 5
Loading 